In [1]:
from nanover.recording import NanoverRecordingReader
from tutorials.experiments.keyframes import extract_keyframes, TargetGroup

IN_PATH = "knot-tying-checkpoints-2.nanover.zip"
# IN_PATH = "trivial-checkpoints.nanover.zip"
IN_PATH = "knot-tying-checkpoints-better.nanover.zip"
# IN_PATH = "nanotube-keyframes.nanover.zip"

with NanoverRecordingReader.from_path(IN_PATH) as reader:
    KEYFRAMES = extract_keyframes(reader)

In [2]:
## Setup runner
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
# simulation = OpenMMSimulation.from_xml_path("nanotube-methane.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="checkpoint replay")
imd_runner.load(0)

In [3]:
simulation.use_pbc_wrapping = False

In [4]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [5]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

In [6]:
from smearmd import SmearAgent

KEYFRAME_INDEX = 0
AGENT: SmearAgent | None = None
SELECTIONS = []

def update_error(errors: list[tuple[TargetGroup, float]]):
    try:
        for i, (target, error) in enumerate(errors):
            error = min(1.0, error)

            with SELECTIONS[i].modify() as selection:
                selection.renderer = {
                    "color": [1.0, 1.0-error, 1.0-error],
                }
    except Exception as e:
        with output:
            print(e)


def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = min(len(KEYFRAMES) - 1, KEYFRAME_INDEX + 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_match(keyframe):
    global AGENT

    if AGENT is None:
        AGENT = SmearAgent.from_runner(imd_runner)
        AGENT.speed = .5 #.1 / 30

    for selection in SELECTIONS:
        client.remove_selection(selection)
    SELECTIONS.clear()
    for i, target in enumerate(keyframe.targets):
        SELECTIONS.append(client.create_selection(f"{i}", target.particles))

    AGENT.set_keyframe(keyframe)
    AGENT.start()

    AGENT.update.add_callback(update_error)


def smear_cancel():
    global AGENT, KEYFRAME_INDEX

    if AGENT is None:
        return

    AGENT.close()
    AGENT = None
    KEYFRAME_INDEX = 0

    notify_all(f"RESETTING AGENT")


imd_runner.app_server.register_command("user/smear/next", smear_next)
imd_runner.app_server.register_command("user/smear/back", smear_back)
imd_runner.app_server.register_command("user/smear/cancel", smear_cancel)

In [7]:
from ipywidgets import Output
output = Output()
output

Output()

In [8]:
with output:
    print("test")

In [9]:
from nanover.jupyter.nglclient import NGLClient

client = NGLClient.from_runner(imd_runner)
client.view

NGLWidget()

In [10]:
from nanover.jupyter.controls import show_runner_controls

show_runner_controls(imd_runner)

Button(description='Close Server', style=ButtonStyle())

interactive(children=(Dropdown(description='Force Type', index=1, options=(('Gaussian', 'gaussian'), ('Harmoni…

interactive(children=(IntSlider(value=100, description='Force Scale', max=1000, min=1), Output()), _dom_classe…

interactive(children=(FloatSlider(value=0.4, description='Force Range', max=2.0, min=0.1, step=0.05), Output()…

interactive(children=(FloatSlider(value=1.0, description='Passthrough', max=1.0), Output()), _dom_classes=('wi…

In [11]:
return

AGENT = SmearAgent.from_runner(imd_runner)
AGENT.set_keyframe(KEYFRAMES[1])

import time

for i in range(10):
    frame = imd_runner.app_server.frame_publisher.current_frame
    AGENT.update_interactions(full_frame=frame, frame_update=frame)
    time.sleep(1/30)


SyntaxError: 'return' outside function (1023570294.py, line 1)

In [ ]:
import numpy as np

a = np.array([0, 1, 2]).reshape(-1, 1)
print(a)
print(np.clip(a, max=1))